# V3 baseline — test mọi checkpoint trên GPT-3.5 Mixed 0%

Build duy nhất một graph `gpt3.5_mixed_test_split`, sau đó test từng checkpoint `.pt`/`.pth`. Nhiều phiên bản có cùng seed vẫn được chạy riêng và giữ tên checkpoint gốc trong kết quả.

In [ ]:
from pathlib import Path
import re, subprocess

BRANCH='feat/rgcn-rdrop'
REPO_URL='https://github.com/tphan1420/PRDetect-Research.git'
WORK=Path('/kaggle/working/v3_all_checkpoints_gpt35_0pct')
REPO=WORK/'PRDetect-Research'
INPUT_CHECKPOINT_DIR=Path('/kaggle/input/datasets/hahahaa1du23/v3-base')

WORK.mkdir(parents=True, exist_ok=True)
CHECKPOINTS=sorted(p for p in INPUT_CHECKPOINT_DIR.rglob('*') if p.is_file() and p.suffix in {'.pt','.pth'})
assert CHECKPOINTS, f'Không có checkpoint .pt/.pth: {INPUT_CHECKPOINT_DIR}'

def seed_from_name(path):
    years=re.findall(r'(?<!\d)(20\d{2})(?!\d)', path.stem)
    assert years, f'Không tìm được seed trong tên checkpoint: {path.name}'
    return int(years[-1])

def checkpoint_id(index, path):
    safe=re.sub(r'[^A-Za-z0-9._-]+', '_', path.stem)
    return f'{index:02d}_{safe}'

CHECKPOINT_RUNS=[{'id': checkpoint_id(i, p), 'path': p, 'seed': seed_from_name(p)}
                 for i, p in enumerate(CHECKPOINTS, start=1)]
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)],check=True)

for item in CHECKPOINT_RUNS:
    print(f"{item['id']} | seed={item['seed']} | {item['path'].name}")


In [ ]:
%cd {REPO}
!pip install -q -r requirements-v3-eval.txt
!python -m spacy download en_core_web_sm
assert _exit_code == 0

In [ ]:
import argparse, csv
from zipfile import ZipFile, ZIP_DEFLATED
import evaluate_v3_quang as ev

case=ev.Case('table3','Original',0,'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json')
source=REPO/case.source
expected=sum(bool(line.strip()) for line in source.open(encoding='utf-8'))
assert expected > 0

# Build one shared graph. It is reused for every checkpoint, including checkpoints
# with the same training seed.
graph_dir=WORK/'graph_data'; graph_dir.mkdir(exist_ok=True)
graph=graph_dir/f'{case.split}.pkl'
if not graph.exists():
    subprocess.run([__import__('sys').executable, str(REPO/'building_graph.py'), '--file', str(source)], cwd=WORK, check=True)
assert graph.is_file(), graph

results=WORK/'result'; results.mkdir(exist_ok=True)
configs=WORK/'configs'; configs.mkdir(exist_ok=True)
staged_root=WORK/'staged_checkpoints'; staged_root.mkdir(exist_ok=True)
metadata=[]; rows=[]

for item in CHECKPOINT_RUNS:
    # The evaluator validates this canonical filename. A private staging folder
    # makes same-seed checkpoints independent instead of overwriting each other.
    stage=staged_root/item['id']; stage.mkdir(exist_ok=True)
    alias=stage/f"v3_baseline_seed{item['seed']}.pt"
    if not alias.exists() or alias.stat().st_size != item['path'].stat().st_size:
        import shutil; shutil.copy2(item['path'], alias)
    args=argparse.Namespace(variant='baseline', train_domain='hc3', checkpoint_dir=str(stage),
                            seeds=[item['seed']], batch_size=4, struct_mode='low',
                            device='cuda', graph_budget_root=None)
    config=ev.make_config(args)
    config['checkpoint_id']=item['id']
    config['original_checkpoint_filename']=item['path'].name
    config_path=configs/f"{item['id']}.json"; ev.write_json(config_path, config)
    info=ev.inspect_checkpoints(config)['checkpoints'][0]
    info.update(checkpoint_id=item['id'], original_checkpoint_filename=item['path'].name)
    metadata.append(info)
    signature=ev.case_signature(config, case, expected)
    report=results/f"{item['id']}_{case.split}.json"
    ev.run_child('_test', '--config', config_path, '--seed', item['seed'], '--graph', graph,
                 '--report', report, '--expected-samples', expected, '--signature', signature)
    value=ev.read_json(report)
    rows.append({'checkpoint_id': item['id'], 'checkpoint_file': item['path'].name,
                 'seed': item['seed'], **{name: value[name] for name in ev.METRICS},
                 'validation_threshold': value['validation_threshold'],
                 'clipped_edges': value['clipped_edges'], 'checkpoint_sha256': value['checkpoint_sha256']})

fields=list(rows[0])
with (results/'per_checkpoint.csv').open('w', newline='', encoding='utf-8') as f:
    writer=csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows(rows)
ev.write_json(results/'checkpoint_metadata.json', {'checkpoints': metadata, 'environment': ev.environment()})
ev.write_json(results/'manifest.json', {'case': case.source, 'samples': expected, 'threshold': 0.5,
                                        'checkpoints': [{'id': x['id'], 'file': x['path'].name, 'seed': x['seed']} for x in CHECKPOINT_RUNS]})
archive=WORK/'v3_baseline_all_checkpoints_gpt35_0pct.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as out:
    for p in sorted(results.iterdir()): out.write(p, f'result/{p.name}')
    out.write(REPO/'evaluate_v3_quang.py', 'evaluate_v3_quang.py')
print('ZIP:', archive)


In [ ]:
from IPython.display import FileLink, display
archive=WORK/'v3_baseline_all_checkpoints_gpt35_0pct.zip'
assert archive.is_file(), archive
display(FileLink(str(archive)))